In [1]:
!pip install datasets transformers pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 11.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found exist

In [2]:
import re
import torch
import torch.nn as nn
import pennylane as qml
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import math
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
batch_size = 4
epochs = 10

n_qubits = 8          # 🔥 increased
n_layers = 4          # 🔥 deeper
n_steps = 2
K = 2

In [4]:
dataset = load_dataset("gsm8k", "main")

train_data = [dataset["train"][i] for i in range(500)]
test_data = [dataset["test"][i] for i in range(200)]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [5]:
def extract_label(example):
    match = re.search(r"#### (\d+)", example["answer"])
    if match:
        val = int(match.group(1))
        return min(val // 10, 99)
    return 0

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
encoder.eval()

def encode_batch(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = encoder(**inputs)
    return outputs.last_hidden_state[:, 0, :]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def build_encoded_dataset(data):

    encoded = []

    for i in tqdm(range(0, len(data), 16), desc="Encoding"):

        batch = data[i:i+16]
        texts = [x["question"] for x in batch]
        labels = [extract_label(x) for x in batch]

        emb = encode_batch(texts).cpu()

        for j in range(len(batch)):
            encoded.append((emb[j], labels[j]))

    return encoded

train_encoded = build_encoded_dataset(train_data)
test_encoded = build_encoded_dataset(test_data)

Encoding:   0%|          | 0/32 [00:00<?, ?it/s]

Encoding:   0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [9]:
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 100)  # 🔥 FIXED
        )

    def forward(self, x):
        x = x.to(device)
        return self.net(x)

In [10]:
class QuantumV93(nn.Module):
    def __init__(self, K=2):
        super().__init__()

        self.K = K

        # 🔥 better projection (less bottleneck)
        self.proj = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, n_qubits)
        )

        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_steps, n_layers, n_qubits, 3)
        )

        self.refine = nn.Sequential(
            nn.Linear(n_qubits, n_qubits),
            nn.Tanh()
        )

        # 🔥 stronger attention
        self.attn = nn.Sequential(
            nn.Linear(n_qubits, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.decoder = nn.Sequential(
            nn.Linear(n_qubits, 128),
            nn.ReLU(),
            nn.Linear(128, 100)
        )

    def forward(self, x):

        x = x.to(device)

        base_state = torch.tanh(self.proj(x)) * math.pi

        # 🔥 add diversity
        state = torch.stack([
            base_state + 0.1 * torch.randn_like(base_state)
            for _ in range(self.K)
        ], dim=1)

        for step in range(n_steps):

            new_states = []

            for k in range(self.K):

                s = state[:, k, :]

                q_out = torch.stack([
                    torch.stack(qnode(s[i], self.q_weights[step]))
                    for i in range(len(s))
                ])

                q_out = q_out.float().to(device)

                # 🔥 stabilized + residual
                s_new = s + 0.5 * self.refine(q_out)

                s_new = torch.clamp(s_new, -math.pi, math.pi)

                new_states.append(s_new)

            state = torch.stack(new_states, dim=1)

        # 🔥 attention routing
        attn_weights = torch.softmax(self.attn(state), dim=1)
        state = (state * attn_weights).sum(dim=1)

        # 🔥 classical residual connection
        state = state + base_state

        logits = self.decoder(state)

        return logits

In [11]:
def build_loader(encoded_data, batch_size=4):

    for i in range(0, len(encoded_data), batch_size):

        batch = encoded_data[i:i+batch_size]

        x = torch.stack([b[0] for b in batch])
        y = torch.tensor([b[1] for b in batch])

        yield x, y

In [12]:
def train(model, data):

    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)  # 🔥 higher LR
    loss_fn = nn.CrossEntropyLoss()

    total_batches = (len(data) + batch_size - 1) // batch_size

    for epoch in range(epochs):

        model.train()

        total_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(
            build_loader(data, batch_size),
            total=total_batches,
            desc=f"Epoch {epoch+1}"
        )

        for xb, yb in pbar:

            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)

            pbar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "acc": f"{(correct/total)*100:.2f}%"
            })

        print(f"\nEpoch {epoch+1}: Loss={total_loss/total_batches:.4f}, Acc={(correct/total)*100:.2f}%")

In [13]:
def evaluate(model, data):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in build_loader(data, batch_size):

            xb, yb = xb.to(device), yb.to(device)

            preds = model(xb).argmax(dim=1)

            correct += (preds == yb).sum().item()
            total += len(yb)

    return correct / total

In [15]:
models = {
    "Classical": ClassicalModel().to(device),
    "Quantum v9.3": QuantumV93(K=K).to(device)
}

results = {}

for name, model in models.items():

    print(f"\n🚀 Training {name}")

    train(model, train_encoded)

    acc = evaluate(model, test_encoded)

    results[name] = acc

print("\n📊 FINAL RESULTS")

for k, v in results.items():
    print(f"{k}: {v*100:.2f}%")


🚀 Training Classical


Epoch 1:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 1: Loss=3.5117, Acc=16.20%


Epoch 2:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 2: Loss=3.0371, Acc=19.20%


...


Epoch 8: Loss=2.6741, Acc=25.00%


Epoch 9:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 9: Loss=2.5933, Acc=27.60%


Epoch 10:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 10: Loss=2.5184, Acc=28.20%

📊 FINAL RESULTS
Classical: 18.50%
Quantum v9.3: 19.50%
